# 🪨 LithoLens — Rock & Mineral Classifier
### Complete Training Pipeline: Download → Train → Export ONNX

**Before running:** Go to Runtime → Change runtime type → Select **T4 GPU**

Then run cells **one by one in order**. Do not skip any cell.

## CELL 1 — Install everything

In [ ]:
# Install required packages
!pip install kaggle timm onnx onnxruntime torch torchvision --quiet
print('✅ All packages installed')

## CELL 2 — Setup Kaggle API
**Steps to get your kaggle.json:**
1. Go to kaggle.com → Your profile picture → Settings
2. Scroll to API section → Click 'Create New Token'
3. A file called kaggle.json downloads to your computer
4. Run this cell and upload that file when prompted

In [ ]:
from google.colab import files
import os

print('Upload your kaggle.json file...')
uploaded = files.upload()

# Move to correct location
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('✅ Kaggle API configured')

## CELL 3 — Download Datasets

In [ ]:
import os

os.makedirs('/content/data', exist_ok=True)
os.chdir('/content/data')

print('Downloading minerals dataset...')
!kaggle datasets download -d asiedubrempong/minerals-identification-dataset --unzip -q

print('Downloading rocks dataset...')
!kaggle datasets download -d salmaneunus/rock-classification --unzip -q

print('Downloading rocks dataset 2...')
!kaggle datasets download -d neelgajare/rocks-dataset --unzip -q

print('✅ All datasets downloaded')
!find /content/data -type d | head -30

## CELL 4 — Explore What We Downloaded

In [ ]:
import os
from pathlib import Path

# Show all class folders found
data_path = Path('/content/data')

print('=== FOUND CLASSES ===')
all_classes = []
for folder in sorted(data_path.rglob('*')):
    if folder.is_dir():
        images = list(folder.glob('*.jpg')) + list(folder.glob('*.png')) + list(folder.glob('*.jpeg'))
        if len(images) > 10:  # Only show folders with real images
            print(f'{folder.name}: {len(images)} images')
            all_classes.append(folder)

print(f'\nTotal class folders found: {len(all_classes)}')

## CELL 5 — Organize into Clean Dataset Structure

In [ ]:
import shutil
import random
from pathlib import Path

# ============================================
# TARGET CLASSES — Edit this list if needed
# Add/remove based on what the datasets contain
# ============================================
TARGET_CLASSES = [
    # Minerals
    'quartz', 'calcite', 'pyrite', 'feldspar', 'muscovite',
    'gypsum', 'halite', 'magnetite', 'hematite', 'olivine',
    'galena', 'fluorite',
    # Rocks
    'granite', 'basalt', 'sandstone', 'limestone', 'marble',
    'slate', 'obsidian', 'schist'
]

# Create clean output structure
output_base = Path('/content/dataset_clean')
train_dir = output_base / 'train'
val_dir = output_base / 'val'

def organize_dataset(source_root, train_ratio=0.8):
    source_root = Path(source_root)
    matched = 0

    for folder in source_root.rglob('*'):
        if not folder.is_dir():
            continue

        folder_lower = folder.name.lower().replace(' ', '_').replace('-', '_')

        # Try to match to our target classes
        matched_class = None
        for cls in TARGET_CLASSES:
            if cls in folder_lower or folder_lower in cls:
                matched_class = cls
                break

        if not matched_class:
            continue

        # Get all images
        images = []
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
            images.extend(folder.glob(ext))

        if len(images) < 5:
            continue

        # Shuffle and split
        random.shuffle(images)
        split_idx = int(len(images) * train_ratio)
        train_imgs = images[:split_idx]
        val_imgs = images[split_idx:]

        # Create directories
        (train_dir / matched_class).mkdir(parents=True, exist_ok=True)
        (val_dir / matched_class).mkdir(parents=True, exist_ok=True)

        # Copy images
        for i, img in enumerate(train_imgs):
            shutil.copy2(img, train_dir / matched_class / f'{matched_class}_{i:04d}{img.suffix}')
        for i, img in enumerate(val_imgs):
            shutil.copy2(img, val_dir / matched_class / f'{matched_class}_val_{i:04d}{img.suffix}')

        print(f'✅ {matched_class}: {len(train_imgs)} train, {len(val_imgs)} val')
        matched += 1

    return matched

random.seed(42)
count = organize_dataset('/content/data')
print(f'\n✅ Organized {count} classes')

# Show final class list
final_classes = sorted([d.name for d in train_dir.iterdir() if d.is_dir()])
print(f'Final classes ({len(final_classes)}): {final_classes}')

## CELL 6 — Create DataLoaders

In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

IMG_SIZE = 224
BATCH_SIZE = 32

# Augmentation for training (makes model more robust to field conditions)
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),  # Simulate field lighting
    transforms.RandomRotation(45),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = ImageFolder('/content/dataset_clean/train', transform=train_transform)
val_dataset = ImageFolder('/content/dataset_clean/val', transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

NUM_CLASSES = len(train_dataset.classes)
CLASS_NAMES = train_dataset.classes

print(f'✅ Train samples: {len(train_dataset)}')
print(f'✅ Val samples: {len(val_dataset)}')
print(f'✅ Number of classes: {NUM_CLASSES}')
print(f'✅ Classes: {CLASS_NAMES}')

## CELL 7 — Build the Model (EfficientNet-B0)

In [ ]:
import timm
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Load pretrained EfficientNet-B0
# pretrained=True means it already knows shapes, textures, colors
# We just retrain the last layer for our classes
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=NUM_CLASSES)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Model loaded')
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

## CELL 8 — Training Setup

In [ ]:
import torch.optim as optim

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer — AdamW works best for transfer learning
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Learning rate scheduler — reduces LR when stuck
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

EPOCHS = 20  # Enough for good accuracy, not too long

print(f'✅ Training setup ready')
print(f'Epochs: {EPOCHS}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Steps per epoch: {len(train_loader)}')
print(f'Estimated time: ~{EPOCHS * len(train_loader) * 0.15 / 60:.0f} minutes on T4 GPU')

## CELL 9 — TRAIN THE MODEL
**This is the main cell. It will take 20-40 minutes. Let it run.**

In [ ]:
import time

best_val_acc = 0.0
best_model_path = '/content/best_model.pth'
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print('🚀 Starting training...')
print('=' * 60)

for epoch in range(EPOCHS):
    start_time = time.time()

    # ---- TRAINING PHASE ----
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for batch_idx, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()

    # ---- VALIDATION PHASE ----
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    scheduler.step()

    # Calculate metrics
    train_acc = 100. * train_correct / train_total
    val_acc = 100. * val_correct / val_total
    epoch_time = time.time() - start_time

    history['train_loss'].append(train_loss / len(train_loader))
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss / len(val_loader))
    history['val_acc'].append(val_acc)

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        saved = '💾 SAVED'
    else:
        saved = ''

    print(f'Epoch {epoch+1:2d}/{EPOCHS} | '
          f'Train: {train_acc:.1f}% | '
          f'Val: {val_acc:.1f}% | '
          f'Time: {epoch_time:.0f}s {saved}')

print('=' * 60)
print(f'✅ Training complete! Best validation accuracy: {best_val_acc:.1f}%')

## CELL 10 — Plot Training Results

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_acc'], label='Train Accuracy', color='#2196F3')
ax1.plot(history['val_acc'], label='Val Accuracy', color='#4CAF50')
ax1.set_title('Accuracy over Epochs')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy (%)')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history['train_loss'], label='Train Loss', color='#F44336')
ax2.plot(history['val_loss'], label='Val Loss', color='#FF9800')
ax2.set_title('Loss over Epochs')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle(f'LithoLens Training — Best Val Accuracy: {best_val_acc:.1f}%', fontsize=14)
plt.tight_layout()
plt.savefig('/content/training_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot saved')

## CELL 11 — Export to ONNX (Enables Offline Use)

In [ ]:
import torch
import json

# Load best model
model.load_state_dict(torch.load(best_model_path))
model.eval()
model.cpu()

# Export to ONNX
dummy_input = torch.randn(1, 3, 224, 224)
onnx_path = '/content/litholens_model.onnx'

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['image'],
    output_names=['predictions'],
    dynamic_axes={
        'image': {0: 'batch_size'},
        'predictions': {0: 'batch_size'}
    }
)

# Save class names — React app needs this
class_names_path = '/content/class_names.json'
with open(class_names_path, 'w') as f:
    json.dump({'classes': CLASS_NAMES, 'num_classes': NUM_CLASSES}, f, indent=2)

import os
model_size = os.path.getsize(onnx_path) / (1024*1024)
print(f'✅ ONNX model exported: {onnx_path}')
print(f'✅ Model size: {model_size:.1f} MB')
print(f'✅ Class names saved: {CLASS_NAMES}')

## CELL 12 — Verify ONNX Works

In [ ]:
import onnxruntime as ort
import numpy as np

# Load and test the ONNX model
session = ort.InferenceSession('/content/litholens_model.onnx')

# Test with random input
test_input = np.random.randn(1, 3, 224, 224).astype(np.float32)
outputs = session.run(None, {'image': test_input})
predictions = outputs[0][0]

# Get top 3
top3_idx = np.argsort(predictions)[::-1][:3]
print('✅ ONNX model works!')
print('Top 3 predictions on random input:')
for i, idx in enumerate(top3_idx):
    softmax = np.exp(predictions) / np.sum(np.exp(predictions))
    print(f'  {i+1}. {CLASS_NAMES[idx]}: {softmax[idx]*100:.1f}%')

## CELL 13 — Test on a Real Image

In [ ]:
from PIL import Image
import numpy as np
import onnxruntime as ort
from google.colab import files

print('Upload a rock or mineral image to test...')
uploaded = files.upload()

for filename in uploaded.keys():
    # Preprocess
    img = Image.open(filename).convert('RGB')
    img = img.resize((224, 224))
    img_array = np.array(img).astype(np.float32) / 255.0

    # Normalize (same as training)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img_array = (img_array - mean) / std
    img_array = img_array.transpose(2, 0, 1)  # HWC -> CHW
    img_array = img_array[np.newaxis, :].astype(np.float32)  # Add batch dim

    # Run inference
    session = ort.InferenceSession('/content/litholens_model.onnx')
    outputs = session.run(None, {'image': img_array})
    predictions = outputs[0][0]

    # Softmax
    softmax = np.exp(predictions) / np.sum(np.exp(predictions))
    top3_idx = np.argsort(softmax)[::-1][:3]

    print(f'\n🪨 Results for {filename}:')
    print('-' * 40)
    for i, idx in enumerate(top3_idx):
        bar = '█' * int(softmax[idx] * 30)
        print(f'{i+1}. {CLASS_NAMES[idx]:<15} {softmax[idx]*100:5.1f}% {bar}')

    top_confidence = softmax[top3_idx[0]] * 100
    if top_confidence < 75:
        print(f'\n⚠️  Low confidence ({top_confidence:.1f}%) — Q&A would trigger in the app')
    else:
        print(f'\n✅ High confidence ({top_confidence:.1f}%) — Direct result in the app')

## CELL 14 — Download Everything You Need

In [ ]:
from google.colab import files
import json

print('Downloading model files...')
print('You need these 2 files for the React app:')
print('1. litholens_model.onnx — the AI model')
print('2. class_names.json — the class labels')
print('')

files.download('/content/litholens_model.onnx')
files.download('/content/class_names.json')
files.download('/content/training_results.png')  # For your pitch deck

print('✅ Download started!')
print('')
print('Place these files in your React app at: /public/model/')
print('  public/')
print('  └── model/')
print('      ├── litholens_model.onnx')
print('      └── class_names.json')

## ✅ Done!

You now have:
- `litholens_model.onnx` — the trained model, runs in browser
- `class_names.json` — class labels for the app
- `training_results.png` — accuracy chart for your pitch deck

**Next step:** Put these files in your React app's `/public/model/` folder and run the app.